In [1]:
# Install required libraries
!pip install chromadb sentence-transformers pypdf transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.4/333.4 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    F

In [2]:
import os

# Read PDF
from pypdf import PdfReader

# Embedding model
from sentence_transformers import SentenceTransformer

# Vector database
import chromadb

# LLM pipeline
from transformers import pipeline

In [3]:
print("Loading PDF document...")

reader = PdfReader("ai_research.pdf")

text = ""

for page in reader.pages:
    text += page.extract_text()

print("Document Loaded")
print("Total Characters:", len(text))

print("\nPreview:\n")
print(text[:500])

Loading PDF document...
Document Loaded
Total Characters: 2812

Preview:

Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think, learn, and make decisions. AI systems can analyze large datasets, recognize
patterns, and generate predictions or recommendations. Modern AI applications include healthcare
diagnostics, recommendation systems, autonomous vehicles, and conversational agents.
Machine Learning
Machine Learning is a sub


In [4]:
print("Splitting document into chunks...")

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks Created:", len(chunks))

print("\nExample Chunk:\n")
print(chunks[0])

Splitting document into chunks...
Total Chunks Created: 7

Example Chunk:

Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think, learn, and make decisions. AI systems can analyze large datasets, recognize
patterns, and generate predictions or recommendations. Modern AI applications include healthcare
diagnostics, recommendation systems, autonomous vehicles, and conversational agents.
Machine Learning
Machine Learning is a sub


In [5]:
print("Loading embedding model...")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding model loaded successfully")

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully


In [6]:
print("Creating Chroma vector database...")

chroma_client = chromadb.Client()

collection = chroma_client.create_collection(name="ai_research")

print("Vector database created")

Creating Chroma vector database...
Vector database created


In [7]:
print("Generating embeddings and storing in database...")

for i, chunk in enumerate(chunks):

    embedding = embedding_model.encode(chunk).tolist()

    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All chunks stored in vector database")

Generating embeddings and storing in database...
All chunks stored in vector database


In [8]:
def retrieve_docs(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results['documents'][0]

In [10]:
print("Loading language model...")

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_length=200
)

print("Language model loaded successfully")

Loading language model...


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCa

Language model loaded successfully


In [11]:
def rag_answer(query):

    docs = retrieve_docs(query)

    context = "\n".join(docs)

    prompt = f"""
    Answer the question using the context below.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    result = generator(prompt)

    return result[0]['generated_text']

In [12]:
query = "What is transformer architecture?"

answer = rag_answer(query)

print("Question:", query)
print("\nAnswer:", answer)

Question: What is transformer architecture?

Answer: 
    Answer the question using the context below.

    Context:
    works with multiple
layers to model complex patterns in data. These models are particularly effective in areas such as
computer vision, speech recognition, and natural language processing.
Transformer Architecture
The transformer architecture is a deep learning model introduced in the paper 'Attention Is All You
Need'. It uses a mechanism called self-attention to process sequences of data. Unlike traditional
recurrent models, transformers process all tokens simultaneously, allowing them to captu
 all tokens simultaneously, allowing them to capture long-range
dependencies efficiently. Transformers are widely used in modern language models such as BERT,
GPT, and T5.
Large Language Models
Large Language Models (LLMs) are neural networks trained on massive text datasets to understand
and generate human language. These models rely heavily on the transformer architecture a

In [13]:
while True:

    user_query = input("\nAsk a question (type 'exit' to stop): ")

    if user_query.lower() == "exit":
        break

    response = rag_answer(user_query)

    print("\nAnswer:", response)


Ask a question (type 'exit' to stop): what is AI?

Answer: 
    Answer the question using the context below.

    Context:
    Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think, learn, and make decisions. AI systems can analyze large datasets, recognize
patterns, and generate predictions or recommendations. Modern AI applications include healthcare
diagnostics, recommendation systems, autonomous vehicles, and conversational agents.
Machine Learning
Machine Learning is a sub
agents.
Machine Learning
Machine Learning is a subset of artificial intelligence that focuses on algorithms that improve
automatically through experience. Instead of being explicitly programmed, machine learning models
learn patterns from data. Common types of machine learning include supervised learning,
unsupervised learning, and reinforcement learning.
Deep Learning